# Project Sanjay MK2 -- YOLO Police Detection Training

Train a YOLO model on VisDrone aerial data remapped to 6 police deployment classes:
- `person` (VisDrone pedestrian + people)
- `weapon_person` (supplementary data)
- `vehicle` (VisDrone car/van/truck/bus/bicycle/motor/tricycle)
- `fire` (supplementary data)
- `explosive_device` (supplementary data)
- `crowd` (supplementary data)

**Phase 1:** Train on VisDrone (person + vehicle only)  
**Phase 2:** Merge supplementary datasets for weapon/fire/crowd  
**Phase 3:** Fine-tune on Isaac Sim synthetic data  

After training, validate in simulation:  
```bash
python scripts/validate_model.py --yolo best.pt --all --compare
```

## 0. Setup (Colab)

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q ultralytics

In [ ]:
import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Download VisDrone Dataset

VisDrone2019-DET: 10,209 images from drone cameras.  
10 classes: pedestrian, people, bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor.  
Ultralytics auto-downloads when referenced.

In [ ]:
from ultralytics.data.utils import check_det_dataset

# This triggers the auto-download (~2GB)
dataset_info = check_det_dataset("VisDrone.yaml")
visdrone_root = dataset_info["path"]
print(f"VisDrone downloaded to: {visdrone_root}")

## 2. Remap Labels to Police Classes

VisDrone's 10 classes -> Sanjay's 6 police classes:  
- pedestrian, people -> `0: person`  
- bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor -> `2: vehicle`  
- Classes 1, 3, 4, 5 (weapon_person, fire, explosive_device, crowd) have no VisDrone source yet.

In [ ]:
import os
import shutil
from pathlib import Path

# VisDrone (0-indexed after Ultralytics conversion) -> Police class
VISDRONE_TO_POLICE = {
    0: 0,   # pedestrian -> person
    1: 0,   # people     -> person
    2: 2,   # bicycle    -> vehicle
    3: 2,   # car        -> vehicle
    4: 2,   # van        -> vehicle
    5: 2,   # truck      -> vehicle
    6: 2,   # tricycle   -> vehicle
    7: 2,   # awning-tricycle -> vehicle
    8: 2,   # bus        -> vehicle
    9: 2,   # motor      -> vehicle
}

visdrone_path = Path(visdrone_root)
output_dir = Path("datasets/visdrone_police")

total_remapped = 0
for split in ["train", "val", "test"]:
    # Find source dirs (layout varies by Ultralytics version)
    src_img = visdrone_path / f"VisDrone2019-DET-{split}" / "images"
    src_lbl = visdrone_path / f"VisDrone2019-DET-{split}" / "labels"
    if not src_img.exists():
        src_img = visdrone_path / "images" / split
        src_lbl = visdrone_path / "labels" / split
    if not src_img.exists():
        print(f"  {split}: not found, skipping")
        continue

    dst_img = output_dir / "images" / split
    dst_lbl = output_dir / "labels" / split
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    # Symlink images
    img_count = 0
    for img in src_img.glob("*"):
        if img.suffix.lower() in (".jpg", ".jpeg", ".png"):
            dst = dst_img / img.name
            if not dst.exists():
                try:
                    dst.symlink_to(img.resolve())
                except OSError:
                    shutil.copy2(img, dst)
                img_count += 1

    # Remap labels
    lbl_count = 0
    if src_lbl.exists():
        for lbl in src_lbl.glob("*.txt"):
            lines = []
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls = int(parts[0])
                        new_cls = VISDRONE_TO_POLICE.get(cls)
                        if new_cls is not None:
                            parts[0] = str(new_cls)
                            lines.append(" ".join(parts))
            with open(dst_lbl / lbl.name, "w") as f:
                f.write("\n".join(lines) + "\n" if lines else "")
            lbl_count += 1

    total_remapped += lbl_count
    print(f"  {split}: {img_count} images, {lbl_count} labels remapped")

print(f"\nTotal: {total_remapped} label files remapped")
print(f"Output: {output_dir}")

In [ ]:
# Write dataset YAML
data_yaml = output_dir / "data.yaml"
data_yaml.write_text("""\
path: datasets/visdrone_police
train: images/train
val: images/val
test: images/test

names:
  0: person
  1: weapon_person
  2: vehicle
  3: fire
  4: explosive_device
  5: crowd
""")
print(f"Dataset config written: {data_yaml}")

## 3. Train YOLO

We use YOLO26s (or YOLOv8s / YOLOv11s) as the base model.  
Augmentations are tuned for aerial drone imagery:  
- High rotation (aerial views can be any angle)  
- Vertical + horizontal flips (BEV has no fixed orientation)  
- Scale jitter (simulates altitude variation)  

In [ ]:
# Choose your base model:
#   yolo26s.pt  -- YOLO26 small (recommended, Jan 2026)
#   yolo26n.pt  -- YOLO26 nano  (faster, less accurate)
#   yolo11s.pt  -- YOLOv11 small (mature, well-tested)
#   yolov8s.pt  -- YOLOv8 small  (most documented)

BASE_MODEL = "yolo26s.pt"  # change as needed
EPOCHS = 100
BATCH = -1     # auto batch size based on GPU memory
IMGSZ = 640

model = YOLO(BASE_MODEL)

results = model.train(
    data=str(data_yaml.resolve()),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    project="runs/detect",
    name="sanjay_police",
    patience=20,
    save=True,
    plots=True,
    # Aerial augmentations
    degrees=15.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    scale=0.5,
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
)

## 4. Evaluate

In [ ]:
# Validate on test set
best = Path("runs/detect/sanjay_police/weights/best.pt")
model = YOLO(str(best))
metrics = model.val(data=str(data_yaml.resolve()), split="test")

print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"\nPer-class AP50:")
for i, name in enumerate(model.names.values()):
    if i < len(metrics.box.ap50):
        print(f"  {name:20s} {metrics.box.ap50[i]:.4f}")

In [ ]:
# Visualize predictions on sample images
from IPython.display import Image, display

# Run inference on a few test images
test_imgs = list((output_dir / "images" / "val").glob("*.jpg"))[:5]
results = model.predict(test_imgs, save=True, project="runs/detect", name="preview")

# Show results
for img_path in sorted(Path("runs/detect/preview").glob("*.jpg"))[:5]:
    display(Image(filename=str(img_path), width=600))

## 5. Export for Edge Deployment

In [ ]:
# Export to ONNX (portable, validate with ONNXAdapter)
model.export(format="onnx", imgsz=640, dynamic=True)
print("ONNX exported.")

# Export to TensorRT (run this ON the Jetson, not in Colab)
# model.export(format="engine", half=True, imgsz=640, device=0)

In [ ]:
# Download weights to local machine
from google.colab import files

files.download(str(best))
print(f"\nDownloaded: {best.name}")
print(f"\nNext: validate in simulation:")
print(f"  python scripts/validate_model.py --yolo {best.name} --all --compare")

## 6. (Optional) Merge Supplementary Data & Retrain

After sourcing weapon/fire/crowd datasets from Kaggle or Roboflow,  
convert to YOLO format and merge into the training set.  

Expected layout for supplementary data:
```
supp_data/
  images/
    train/  *.jpg
    val/    *.jpg
  labels/
    train/  *.txt   (YOLO format, using police class indices)
    val/    *.txt
```

Class indices for labels:  
- 1 = weapon_person  
- 3 = fire  
- 4 = explosive_device  
- 5 = crowd  

In [ ]:
# Example: merge a weapon detection dataset
# Uncomment and adjust paths after downloading supplementary data

# supp_dir = Path("datasets/weapon_data")
# for split in ["train", "val"]:
#     src_img = supp_dir / "images" / split
#     src_lbl = supp_dir / "labels" / split
#     if src_img.exists():
#         for f in src_img.glob("*.jpg"):
#             shutil.copy2(f, output_dir / "images" / split / f.name)
#     if src_lbl.exists():
#         for f in src_lbl.glob("*.txt"):
#             shutil.copy2(f, output_dir / "labels" / split / f.name)
#         print(f"  Merged {split} from {supp_dir}")
# 
# # Retrain with merged data
# model = YOLO(BASE_MODEL)
# model.train(data=str(data_yaml), epochs=100, ...)